# 전체 탐지 데이터 준비

먼저 `DETECTOR_DATA_GUIDE.md`의 3번대로 데이터를 `datasets/`에 배치하세요.
저장소의 학습 환경을 커널로 선택합니다.
이 Notebook은 고정 분할을 읽고 DataLoader를 준비합니다. 학습은 시작하지 않습니다.
표본 모델 연결 검사는 `detector_data_smoke_test/smok_test.ipynb`에서 진행합니다.

## 1. 저장소와 실행 환경 확인

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "detector").is_dir() and (p / "detector_data").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("저장소 안에서 Notebook을 실행하세요.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("저장소:", project_root)
print("Python:", sys.executable)


## 2. 전체 라벨·분할·이미지 파일 존재 검사

JSON과 분할 목록의 연결, ID 중복, 그룹 중복, 박스 좌표·면적, 파일 존재를 검사합니다.
이미지 픽셀 전체를 다시 읽는 검사는 아닙니다.

In [ ]:
subprocess.run([sys.executable, "-m", "detector_data", "check"],
               cwd=project_root, check=True)


## 3. Dataset·DataLoader 생성

모든 알약을 탐지 라벨 1로 변환하고 팀원의 검증 함수로 첫 배치를 검사합니다.
확인 후 소비하지 않은 로더를 다시 만들어 반환합니다.
이전 학습 결과가 있는 이름은 피해서 `run_name`을 지정하세요.

In [ ]:
from detector_data.start import prepare_training_inputs

config, device, train_loader, val_loader, label_map, data_summary = (
    prepare_training_inputs(
        run_name="pill_baseline_01", batch_size=8, seed=42, max_epochs=50,
    )
)
print(json.dumps(data_summary, ensure_ascii=False, indent=2))


## 여기까지가 데이터 준비입니다

`train_loader`, `val_loader`를 팀원의 학습 함수에 전달할 수 있습니다.
실제 학습 호출은 `DETECTOR_DATA_GUIDE.md` 4번에 따로 있습니다.
표본 검사를 하려는 경우 학습 함수를 호출하지 마세요.